In [ ]:
# mount drive
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

if Path("/content/drive/MyDrive/training-embedding").exists():
    project_root = "/content/drive/MyDrive/training-embedding"
elif Path("/content/drive/My Drive/training-embedding").exists():
    project_root = "/content/drive/My Drive/training-embedding"
else:
    raise RuntimeError("training-embedding folder not found under mounted Drive.")

ollama_models = f"{project_root}/ollama/models"
print(f"project_root={project_root}")
print(f"ollama_models={ollama_models}")


Mounted at /content/drive
project_root=/content/drive/MyDrive/training-embedding
ollama_models=/content/drive/MyDrive/training-embedding/ollama/models


In [ ]:
# install ollama to session disk and start ollama

%%bash
set -euo pipefail

if [ -d /content/drive/MyDrive/training-embedding ]; then
  PROJECT_ROOT="/content/drive/MyDrive/training-embedding"
elif [ -d "/content/drive/My Drive/training-embedding" ]; then
  PROJECT_ROOT="/content/drive/My Drive/training-embedding"
else
  echo "Could not find training-embedding folder in Drive." >&2
  exit 1
fi

ARCH="$(uname -m)"
case "$ARCH" in
  x86_64) ARCH="amd64" ;;
  aarch64|arm64) ARCH="arm64" ;;
  *) echo "Unsupported architecture: $ARCH" >&2; exit 1 ;;
esac

if ! command -v zstd >/dev/null 2>&1; then
  apt-get update -qq
  apt-get install -y -qq zstd
fi

OLLAMA_MODELS="$PROJECT_ROOT/ollama/models"
RUNTIME_ROOT="/content/ollama-runtime"
RUNTIME_BIN="$RUNTIME_ROOT/bin/ollama"
TARBALL="/content/ollama-linux-${ARCH}.tar.zst"
OLLAMA_LOG="/content/ollama-serve.log"
FORCE_CPU="${FORCE_CPU:-0}"

mkdir -p "$OLLAMA_MODELS/blobs" "$OLLAMA_MODELS/manifests"

if [ ! -s "$TARBALL" ] || ! tar --zstd -tf "$TARBALL" >/dev/null 2>&1; then
  rm -f "$TARBALL"
  curl -fL "https://ollama.com/download/ollama-linux-${ARCH}.tar.zst" -o "$TARBALL"
fi

rm -rf "$RUNTIME_ROOT"
mkdir -p "$RUNTIME_ROOT"
tar --zstd -xf "$TARBALL" -C "$RUNTIME_ROOT"
chmod +x "$RUNTIME_BIN"

pkill -f "$RUNTIME_BIN serve" >/dev/null 2>&1 || true
if [ "$FORCE_CPU" = "1" ]; then
  nohup env OLLAMA_MODELS="$OLLAMA_MODELS" OLLAMA_HOST="127.0.0.1:11434" CUDA_VISIBLE_DEVICES="-1" OLLAMA_LLM_LIBRARY="cpu" \
    "$RUNTIME_BIN" serve >"$OLLAMA_LOG" 2>&1 &
else
  nohup env OLLAMA_MODELS="$OLLAMA_MODELS" OLLAMA_HOST="127.0.0.1:11434" \
    "$RUNTIME_BIN" serve >"$OLLAMA_LOG" 2>&1 &
fi

for i in $(seq 1 40); do
  if curl -sf http://127.0.0.1:11434/api/version >/dev/null; then
    break
  fi
  sleep 1
done

curl -sf http://127.0.0.1:11434/api/version
curl -sf http://127.0.0.1:11434/api/tags


{"version":"0.16.0"}{"models":[{"name":"turkish-gemma:latest","model":"turkish-gemma:latest","modified_at":"2026-02-12T07:24:29Z","size":4761781606,"digest":"6c35cb47d01a769fe651412077a49e450b354c9e7f6b96f67cace4b3b3d0515f","details":{"parent_model":"","format":"gguf","family":"gemma2","families":["gemma2"],"parameter_size":"9.2B","quantization_level":"Q3_K_M"}}]}

In [ ]:
import json
import requests

resp = requests.get("http://127.0.0.1:11434/api/tags", timeout=30)
resp.raise_for_status()
tags = resp.json()
print(json.dumps(tags, indent=2, ensure_ascii=False))


{
  "models": [
    {
      "name": "turkish-gemma:latest",
      "model": "turkish-gemma:latest",
      "modified_at": "2026-02-12T07:24:29Z",
      "size": 4761781606,
      "digest": "6c35cb47d01a769fe651412077a49e450b354c9e7f6b96f67cace4b3b3d0515f",
      "details": {
        "parent_model": "",
        "format": "gguf",
        "family": "gemma2",
        "families": [
          "gemma2"
        ],
        "parameter_size": "9.2B",
        "quantization_level": "Q3_K_M"
      }
    }
  ]
}


In [ ]:
# register quantized GGUF into Ollama (idempotent)
import json
import os
import subprocess
import tempfile
from pathlib import Path

MODEL_NAME = "turkish-gemma-v01-q4km"
GGUF_PATH = Path(project_root) / "ollama" / "downloads" / "Turkish-Gemma-9b-v0.1.Q4_K_M.gguf"
OLLAMA_BIN = Path("/content/ollama-runtime/bin/ollama")
OLLAMA_HOST = "127.0.0.1:11434"

if not OLLAMA_BIN.exists():
    raise FileNotFoundError(f"Ollama binary not found: {OLLAMA_BIN}")
if not GGUF_PATH.exists():
    raise FileNotFoundError(f"GGUF not found for registration: {GGUF_PATH}")

env = os.environ.copy()
env["OLLAMA_MODELS"] = ollama_models
env["OLLAMA_HOST"] = OLLAMA_HOST

tags_raw = subprocess.check_output([str(OLLAMA_BIN), "list"], env=env, text=True)
already_registered = any(
    line.strip().startswith(f"{MODEL_NAME}:") or line.strip().startswith(MODEL_NAME + " ")
    for line in tags_raw.splitlines()
)

if already_registered:
    print(f"Model already registered: {MODEL_NAME}")
else:
    with tempfile.NamedTemporaryFile("w", suffix=".modelfile", delete=False) as tmp:
        modelfile_path = Path(tmp.name)
        tmp.write(f"FROM {GGUF_PATH}\n")
        tmp.write("PARAMETER num_ctx 2048\n")

    try:
        subprocess.run(
            [str(OLLAMA_BIN), "create", MODEL_NAME, "-f", str(modelfile_path)],
            env=env,
            check=True,
        )
        print(f"Registered model: {MODEL_NAME}")
    finally:
        modelfile_path.unlink(missing_ok=True)

print("\nCurrent models:")
print(subprocess.check_output([str(OLLAMA_BIN), "list"], env=env, text=True))


Registered model: turkish-gemma-v01-q4km

Current models:
NAME                             ID              SIZE      MODIFIED               
turkish-gemma-v01-q4km:latest    7a637ef3a0db    5.8 GB    Less than a second ago    
turkish-gemma:latest             6c35cb47d01a    4.8 GB    12 hours ago              



In [9]:
# Optional cleanup: remove source GGUF after successful Ollama registration
from pathlib import Path

gguf_path = Path(project_root) / "ollama" / "downloads" / "Turkish-Gemma-9b-v0.1.Q4_K_M.gguf"

if gguf_path.exists():
    gguf_path.unlink()
    print(f"Deleted source GGUF: {gguf_path}")
else:
    print(f"Source GGUF not found (already removed): {gguf_path}")


Deleted source GGUF: /content/drive/MyDrive/training-embedding/ollama/downloads/Turkish-Gemma-9b-v0.1.Q4_K_M.gguf


In [ ]:
# load/warm up selected model into Ollama runtime
import requests

MODEL_NAME = globals().get("MODEL_NAME", "turkish-gemma-v01-q4km")

warmup_payload = {
    "model": MODEL_NAME,
    "prompt": "Merhaba",
    "stream": False,
    "keep_alive": "30m",
    "options": {"num_predict": 1, "temperature": 0.0, "num_ctx": 2048},
}

resp = requests.post("http://127.0.0.1:11434/api/generate", json=warmup_payload, timeout=600)
resp.raise_for_status()
print(f"Warm-up done for model: {MODEL_NAME}")


Warm-up done for model: turkish-gemma-v01-q4km


In [ ]:
import requests

MODEL_NAME = globals().get("MODEL_NAME", "turkish-gemma-v01-q4km")
PROMPT = "Merhaba!"

payload = {
    "model": MODEL_NAME,
    "prompt": PROMPT,
    "stream": False,
    "options": {"temperature": 0.01, "num_ctx": 2048},
}

resp = requests.post("http://127.0.0.1:11434/api/generate", json=payload, timeout=600)
resp.raise_for_status()
data = resp.json()
print("Model:", MODEL_NAME)
print("Prompt:", PROMPT)
print("\nResponse:\n")
print((data.get("response") or "").strip())


ReadTimeout: HTTPConnectionPool(host='127.0.0.1', port=11434): Read timed out. (read timeout=600)